# 🧬 Official BCR Prediction Pipeline
## Honest Nested-CV with Stability Selection + Elastic Net

**This is the ONLY official pipeline for BCR prediction.** All other notebooks/scripts have been archived.

### Key Features:
1. ✅ **No Data Leakage**: Feature selection happens INSIDE each fold of nested-CV
2. ✅ **Stability Selection**: Bootstrap L1 instead of PSO wrapper
3. ✅ **Calibrated Model**: Elastic Net with isotonic calibration
4. ✅ **Clinical Baseline**: Reference model to beat
5. ✅ **Cross-Platform Validation**: Harmonized external validation

---

In [ ]:
# Setup
import sys
sys.path.append('/workspace/core')

import pandas as pd
import numpy as np
import logging
from src.io import setup_logging, logger
from src.data_loaders import load_tcga_prad_bcr
from src.preprocessing import preprocess_pipeline
from src.improved_pipeline import (
    stability_selection,
    build_elastic_net,
    nested_cv_with_selection,
    evaluate_clinical_baseline,
    CLINICAL_BASELINE_FEATURES
)
from src.validation import normalize_cross_platform

setup_logging(level=logging.INFO)
RANDOM_STATE = 42

## Step 1: Load and Prepare Data

In [ ]:
# Load TCGA-PRAD data
clinical, rna_seq, bcr_labels = load_tcga_prad_bcr()

logger.info(f"Loaded {len(clinical)} patients, {rna_seq.shape[1]} genes")
logger.info(f"BCR positive rate: {bcr_labels.mean():.2%}")

# Merge clinical + RNA-seq
merged = clinical.merge(rna_seq, left_index=True, right_index=True, how='inner')
X = merged.drop(columns=[c for c in ['BCR', 'days_to_bcr'] if c in merged.columns], errors='ignore')
y = bcr_labels.reindex(X.index).dropna()
X = X.loc[y.index]

logger.info(f"Final dataset: {X.shape[0]} samples, {X.shape[1]} features")

## Step 2: Preprocessing (INSIDE CV - done automatically in pipeline)

In [ ]:
# Apply preprocessing ONCE before CV (imputation, log-transform, scaling)
# Note: In improved_pipeline, this happens inside each fold to avoid leakage
X_clean = X.dropna(axis=1, thresh=len(X)*0.8)  # Remove genes with >20% missing
X_clean = X_clean.fillna(X_clean.median())  # Simple imputation

# Log-transform RNA-seq (TPM/FPKM)
gene_cols = [c for c in X_clean.columns if c not in CLINICAL_BASELINE_FEATURES]
X_clean[gene_cols] = np.log2(X_clean[gene_cols] + 1)

logger.info(f"After cleaning: {X_clean.shape[1]} features")

## Step 3: Clinical Baseline Model

In [ ]:
# Evaluate clinical-only baseline
clinical_results = evaluate_clinical_baseline(X_clean, y, random_state=RANDOM_STATE)

print("\n" + "="*60)
print("📊 CLINICAL BASELINE RESULTS")
print("="*60)
print(f"Features used: {clinical_results['n_features']}")
print(f"AUC: {clinical_results['clinical_auc_mean']:.3f} ± {clinical_results['clinical_auc_std']:.3f}")
print(f"AP:  {clinical_results['clinical_ap_mean']:.3f}")
print("="*60)

## Step 4: Genomic Model with Honest Nested-CV

In [ ]:
# Run honest nested-CV with stability selection INSIDE each fold
genomic_results = nested_cv_with_selection(
    X=X_clean,
    y=y,
    n_splits=5,
    n_repeats=3,
    n_boot=50,
    stability_threshold=0.6,
    max_features=12,
    random_state=RANDOM_STATE
)

summary = genomic_results.summary()

print("\n" + "="*60)
print("🧬 GENOMIC MODEL RESULTS (Honest Nested-CV)")
print("="*60)
print(f"Outer folds: {summary['n_folds']}")
print(f"AUC: {summary['mean_outer_auc']:.3f} ± {summary['std_outer_auc']:.3f}")
print(f"AP:  {summary['mean_outer_ap']:.3f}")
print(f"Clinical baseline AUC: {clinical_results['clinical_auc_mean']:.3f}")
print(f"Improvement: {summary['mean_outer_auc'] - clinical_results['clinical_auc_mean']:+.3f}")
print("="*60)

# Show most stable features
if genomic_results.feature_stability is not None:
    print("\nTop 10 Most Stable Features:")
    print(genomic_results.feature_stability.head(10))

## Step 5: Train Final Model on Full Training Data

In [ ]:
# Final stability selection on full training data
_, final_features = stability_selection(
    X=X_clean,
    y=y,
    n_boot=100,
    threshold=0.6,
    max_features=12,
    random_state=RANDOM_STATE
)

logger.info(f"Final selected features ({len(final_features)}): {final_features}")

# Train calibrated model
final_model = build_elastic_net(random_state=RANDOM_STATE)
final_model.fit(X_clean[final_features], y)

logger.info("Final model trained and calibrated")

## Step 6: Save Model and Results

In [ ]:
import pickle
import json

# Save model
with open('/workspace/core/models/final_bcr_model.pkl', 'wb') as f:
    pickle.dump({
        'model': final_model,
        'features': final_features,
        'clinical_auc': clinical_results['clinical_auc_mean'],
        'genomic_auc': summary['mean_outer_auc'],
        'genomic_auc_std': summary['std_outer_auc']
    }, f)

# Save results summary
results = {
    'clinical_baseline': clinical_results,
    'genomic_model': summary,
    'selected_features': final_features,
    'feature_stability': genomic_results.feature_stability.to_dict() if genomic_results.feature_stability is not None else None
}

with open('/workspace/core/results/pipeline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

logger.info("Model and results saved to /workspace/core/models/ and /workspace/core/results/")

## Summary

✅ **Completed:**
- Honest nested-CV with feature selection inside each fold
- Clinical baseline established
- Genomic model trained with stability selection
- Model calibrated for interpretable probabilities

📌 **Next Steps:**
1. Run `03_External_Validation.ipynb` for cross-platform validation
2. Run `04_Explainability_and_Baseline.ipynb` for SHAP analysis
3. Review results in `/workspace/core/results/pipeline_results.json`